# 💻 CLI and YAML Pipelines

CircuitKit provides a full CLI for terminal and CI/CD workflows. Every Python API operation has a CLI equivalent.

This notebook demonstrates:
- **Section A:** Discovery via `circuitkit discover`
- **Section B:** Inspecting artifacts with `circuitkit inspect`
- **Section C:** Pruning and exporting via CLI
- **Section D:** YAML task configs with `circuitkit discover-yaml`
- **Section E:** Full YAML pipeline with `circuitkit run`
- **Section F:** Data utilities

- **Model:** GPT-2 (CPU-friendly)
- **Runtime:** ~5 min

---

## Setup

In [ ]:
# For Colab: uncomment and run the setup cells from 00_colab_setup.ipynb
# For local: pip install circuitkit

import os
os.makedirs('./results/cli_demo', exist_ok=True)

In [ ]:
# Verify CLI is installed
!circuitkit --help

---
## Section A: Discovery via CLI

The `circuitkit discover` command mirrors `Pipeline.discover()`.

In [ ]:
!circuitkit discover \
    --model gpt2 \
    --algorithm eap-ig \
    --task ioi \
    --level node \
    --scope both \
    --sparsity 0.3 \
    --batch-size 4 \
    --num-examples 32 \
    --ig-steps 5 \
    --output ./results/cli_demo/circuit.pt

---
## Section B: Inspect the Artifact

`circuitkit inspect` loads a `.pt` artifact and prints a structured summary.

In [ ]:
!circuitkit inspect ./results/cli_demo/circuit.pt

---
## Section C: Prune and Export via CLI

The CLI `prune` command loads the model, applies the circuit-guided pruning mask, and writes a HuggingFace checkpoint in one shot.

In [ ]:
!circuitkit prune \
    --model gpt2 \
    --artifact ./results/cli_demo/circuit.pt \
    --sparsity 0.3 \
    --scope both \
    --precision float32 \
    --output ./results/cli_demo/pruned_checkpoint

In [ ]:
# List the exported checkpoint files
!ls -la ./results/cli_demo/pruned_checkpoint/

---
## Section D: YAML Task Discovery

For custom data, you can define a task in a YAML file and discover circuits without writing Python code. `YAMLTaskLoader` expects three blocks:

- `source` — `type` (`csv`/`jsonl`/`hf`) and `path` (or `dataset_id` for `hf`)
- `schema` — maps `prompt` and `answer` (required) to column names; `choices` is optional
- `corruption` — optional. `strategy` is one of `entity_swap`, `token_swap`, `paraphrase`, `distractor`, `role_swap`, with an optional `config` block
- `metric` — optional, defaults to `logit_diff` (required for EAP-family algorithms; `accuracy` is non-differentiable and reporting-only)

First, create a sample CSV and YAML config inline. We use `token_swap` corruption to automatically generate corrupted prompts from the clean ones.

In [ ]:
%%writefile ./results/cli_demo/sample_task.csv
question,answer
"The capital of France is"," Paris"
"The capital of Japan is"," Tokyo"
"The capital of Italy is"," Rome"
"The capital of Germany is"," Berlin"
"The capital of Spain is"," Madrid"
"The capital of Egypt is"," Cairo"
"The capital of Canada is"," Ottawa"
"The capital of Brazil is"," Brasilia"

In [ ]:
%%writefile ./results/cli_demo/sample_task.yaml
name: capital_cities
source:
  type: csv
  path: ./results/cli_demo/sample_task.csv
schema:
  prompt: question
  answer: answer
corruption:
  strategy: token_swap
metric: logit_diff

In [ ]:
!circuitkit discover-yaml \
    --model gpt2 \
    --task-yaml ./results/cli_demo/sample_task.yaml \
    --algorithm eap-ig \
    --level node \
    --sparsity 0.2 \
    --num-examples 8 \
    --output ./results/cli_demo/yaml_circuit.pt

---
## Section E: Full YAML Pipeline

The `circuitkit run` command executes a complete pipeline from a single YAML config. It supports: model, discovery, evaluate, applications, export, benchmark, and visualize.

Key YAML semantics:
- `evaluate.enabled` defaults to `True` when the `evaluate:` key is present
- `applications` is a list of intervention steps
- Steps after discovery are non-fatal — failures print warnings and continue

In [ ]:
%%writefile ./results/cli_demo/pipeline.yaml
model: gpt2
task: ioi
precision: float32
output_dir: ./results/cli_demo/pipeline_out

discovery:
  algorithm: eap-ig
  level: node
  sparsity: 0.3
  n_examples: 32
  batch_size: 4

evaluate:
  pillars:
    - patching
    - ablation
  n_examples: 32

applications:
  - type: prune
    sparsity: 0.3
    scope: both

export:
  path: ./results/cli_demo/pipeline_out/checkpoint
  intervention: pruning

visualize:
  mode: graph
  output: ./results/cli_demo/pipeline_out/circuit.html

In [ ]:
!circuitkit run ./results/cli_demo/pipeline.yaml

---
## Section F: Data Utilities

The `circuitkit data` subcommands help prepare and validate custom datasets.

In [ ]:
# List available data shapes
!circuitkit data shapes

In [ ]:
# List available data strategies
!circuitkit data strategies

---
## CLI Command Reference

| Command | Description |
|---------|-------------|
| `circuitkit discover` | Run circuit discovery on a built-in task |
| `circuitkit discover-yaml` | Discover using a YAML task config |
| `circuitkit evaluate` | Evaluate a circuit artifact |
| `circuitkit inspect` | Print artifact summary |
| `circuitkit prune` | Prune + export in one shot |
| `circuitkit quantize` | Quantize + export in one shot |
| `circuitkit export` | Export an intervened model |
| `circuitkit run` | Full pipeline from YAML |
| `circuitkit benchmark` | Run lm-eval harness |
| `circuitkit list-models` | List supported models |
| `circuitkit discover-smart` | Memory-aware discovery |
| `circuitkit data *` | Data preparation utilities |